In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

c:\Users\HP\numpy\diabetes-prediction


In [3]:

import pandas as pd
from pathlib import Path
from src.preprocessing.feature_engineering import create_features
from src.preprocessing.preprocessor import build_preprocessor

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

X_train = train_df.drop(columns=["glyhb"])
y_train = train_df["glyhb"]

X_test = test_df.drop(columns=["glyhb"])
y_test = test_df["glyhb"]

# feature engineering
X_train = create_features(X_train)
X_test = create_features(X_test)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)
print("Training missing values:")
print(X_train.isnull().sum())

print("\nTesting missing values:")
print(X_test.isnull().sum())

Training features: (312, 17)
Testing features: (78, 17)
Training target: (312,)
Testing target: (78,)
Training missing values:
chol                1
stab.glu            0
hdl                 1
ratio               1
location            0
age                 0
gender              0
height              4
weight              0
frame              10
bp.1s               5
bp.1d               5
waist               2
hip                 2
time.ppn            3
bmi                 4
waist_hip_ratio     2
dtype: int64

Testing missing values:
chol               0
stab.glu           0
hdl                0
ratio              0
location           0
age                0
gender             0
height             1
weight             1
frame              1
bp.1s              0
bp.1d              0
waist              0
hip                0
time.ppn           0
bmi                2
waist_hip_ratio    0
dtype: int64


In [4]:
import joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

PREPROCESSOR_DIR = PROJECT_ROOT / "models" / "preprocessing"
PREPROCESSOR_DIR.mkdir(parents=True, exist_ok=True)
preprocessor = build_preprocessor(X_train)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)


joblib.dump(
    preprocessor,
    PREPROCESSOR_DIR / "regression_preprocessor.pkl"
)

print("Regression preprocessor saved successfully!")

Processed training shape: (312, 21)
Processed testing shape: (78, 21)
Regression preprocessor saved successfully!


In [7]:
import numpy as np
print("Training missing values:", np.isnan(X_train_processed).sum())
print("Testing missing values:", np.isnan(X_test_processed).sum())

Training missing values: 0
Testing missing values: 0


# Data Preprocessing

## Objective
Prepare the training and testing features for machine learning through feature engineering, missing-value handling, encoding, and scaling.

## Data Preparation
- Loaded the previously saved `train.csv` and `test.csv` datasets.
- Separated the input features from the target variable (`glyhb`).
- Retained the existing 80:20 train-test split.

## Feature Engineering
- Created two additional features: **BMI** from weight and height, and **waist-to-hip ratio** from waist and hip measurements.
- Retained the original features and applied the same transformations to both training and testing data.

## Data Transformation
- Used median imputation for missing numerical values and most-frequent imputation for categorical values.
- Applied `StandardScaler` to numerical features and `OneHotEncoder` to categorical features.
- Combined these operations using `ColumnTransformer` within a preprocessing pipeline.

## Applying the Preprocessing Pipeline
- Built the preprocessing pipeline using `build_preprocessor()`.
- Used `fit_transform()` on training data to learn preprocessing parameters and transform the features.
- Used `transform()` on testing data to apply the same learned parameters, preventing data leakage.

## Results
| Dataset | Before preprocessing | After preprocessing |
|---|---:|---:|
| Training | 312 × 17 | 312 × 21 |
| Testing | 78 × 17 | 78 × 21 |

- Feature engineering increased the input features from 15 to 17.
- One-hot encoding increased the processed feature count to 21.
- The number of records remained unchanged.
- Both processed datasets contained **zero missing values**.

## Conclusion
Feature engineering and preprocessing were successfully completed. The resulting training and testing features are ready for regression experiments.

## Next Step
Proceed to **Regression Experiments** to train and compare different algorithms for predicting GlyHb percentages.

In [11]:
import pandas as pd

# Get the names of the processed features
feature_names = preprocessor.get_feature_names_out()

# Convert processed features into DataFrames
X_train_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_test_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)

# Add the target column
X_train_df["glyhb"] = y_train.reset_index(drop=True)
X_test_df["glyhb"] = y_test.reset_index(drop=True)

# Save processed datasets
X_train_df.to_csv(PROCESSED_DIR / "train_processed.csv", index=False)
X_test_df.to_csv(PROCESSED_DIR / "test_processed.csv", index=False)

print("Processed training and testing data saved successfully.")

Processed training and testing data saved successfully.
